### RAG Pipeline  - Data ingettion to Vector DB 

In [9]:
import os 
from langchain_community.document_loaders import PyMuPDFLoader,PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [12]:
### read all the pdf from the folders 

def process_all_pdf(pdf_directory):
    all_document = []
    pdf_dir = Path(pdf_directory)
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    print(f"Found {len(pdf_files)} PDF files to process")

    for pdf_file in pdf_files:
        print(f"\n Processing : {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
        
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'

            all_document.extend(documents)
            print(f" Loaded {len(documents)} Pages")

        except Exception as e :
            print(f"Error : {e}")

    print(f"\n Total documents loaded  : {len(all_document)}")
    return all_document
all_pdf_documents = process_all_pdf("../documents/pdf")


Found 1 PDF files to process

 Processing : LLM_Research_Report_Dhruvkumar.pdf
 Loaded 30 Pages

 Total documents loaded  : 30


In [13]:
### Text splitting get into chunks 

def split_documents(documents, chunk_size=1000, chunk_overLap=200):
    Text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overLap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = Text_splitter.split_documents(documents)
    print(f"\n Total documents after splitting  : {len(split_docs)}")

    #show example of the chunks
    if split_docs:
        print(f"\n Example of chunks")
        print(f" Content : {split_docs[0].page_content[:200]} ...")
        print(f" Metadata : {split_docs[0].metadata}")

    return split_docs

In [14]:
chunks = split_documents(all_pdf_documents, chunk_size=1000, chunk_overLap=200)


 Total documents after splitting  : 48

 Example of chunks
 Content : Large Language Models (LLMs):
Architecture, Training, Comparison
and Applications
Prepared By: Dhruvkumar Dobariya
A Comprehensive Professional Research Report ...
 Metadata : {'producer': 'WeasyPrint 62.3', 'creator': 'PyPDF', 'creationdate': '', 'title': 'LLM Research Report', 'source': '..\\documents\\pdf\\LLM_Research_Report_Dhruvkumar.pdf', 'total_pages': 30, 'page': 0, 'page_label': '1', 'source_file': 'LLM_Research_Report_Dhruvkumar.pdf', 'file_type': 'pdf'}


# Embeddings and Vector DB

In [20]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
import uuid
from chromadb.config import Settings
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity


In [28]:
class EmbeddingManager:
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        self.model_name = model_name
        self.model = None
        self.load_model()

    def load_model(self):
        try:
            print(f"Loading embedding model... {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully : {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model: {e}")
            raise 

    def generating_embeddings(self, texts: List[str]) -> np.ndarray:
        if not self.model:
            raise ValueError("Model is not loaded. Please load the model before generating embeddings.")
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts,  show_progress_bar=True)
        print(f"Embeddings generated successfully. Shape: {embeddings.shape}")
        return embeddings

Embedding_Manager = EmbeddingManager()
Embedding_Manager


Loading embedding model... all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Model loaded successfully : 384


C:\Users\dobar\AppData\Local\Temp\ipykernel_10736\3738717109.py:11: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded successfully : {self.model.get_sentence_embedding_dimension()}")


# Vector Store

In [32]:
class VectorStore:
    def __init__(self, collection_name: str = "pdf_documents",persist_directory: str = "../data/vector_store"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self._initialize_store()

    def _initialize_store(self):
        try:
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            self.collection = self.client.get_or_create_collection(name=self.collection_name,metadata={"dictionary": "PDF Documents embeddings for RAG"})
            print(f"Vector store initialized successfully. Collection name: {self.collection_name}")
            print(f"Existing documents in the collection: {self.collection.count()}")
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents and embeddings must match.")

        print(f"Adding {len(documents)} documents to the vector store...")
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i , (doc,embedding) in enumerate(zip(documents, embeddings)):
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_legth'] = len(doc.page_content)
            metadatas.append(metadata)
            documents_text.append(doc.page_content)
            embeddings_list.append(embedding.tolist())

        try:
            self.collection.add(
                ids=ids,
                metadatas=metadatas,
                documents=documents_text,
                embeddings=embeddings_list
            )
            print(f"Successfully added {len(documents)} documents to the vector store.")
            print(f"Total documents in the collection after addition: {self.collection.count()}")
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vector_store = VectorStore()
vector_store


Vector store initialized successfully. Collection name: pdf_documents
Existing documents in the collection: 0


In [35]:
### covert the chunks into the embeddings and add to the vector store
texts = [doc.page_content for doc in chunks]

# Generate embeddings for the chunks
embeddings = Embedding_Manager.generating_embeddings(texts)

# store in the vector database
vector_store.add_documents(chunks, embeddings)




Generating embeddings for 48 texts...


Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Embeddings generated successfully. Shape: (48, 384)
Adding 48 documents to the vector store...
Successfully added 48 documents to the vector store.
Total documents in the collection after addition: 48


# Retrive Function 

In [36]:
class RAGRetriever:
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        print(f"Retrieving documents for query: '{query}' with top_k={top_k} and score_threshold={score_threshold}")
        query_embedding = self.embedding_manager.generating_embeddings([query])[0]

        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            retrieved_docs = []
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]

                for i, (doc, metadata, distance, doc_id) in enumerate(zip(documents, metadatas, distances, ids)):
                    score = 1 - distance
                    if score >= score_threshold:
                        retrieved_docs.append({
                            "id": doc_id,
                            "content": doc,
                            "metadata": metadata,
                            "score": score,
                            "distance": distance,
                            "rank": i + 1
                        })
                print(f"Retrieved {len(retrieved_docs)} documents after applying score threshold.")
            else:
                print("No documents retrieved from the vector store.")
            
            return retrieved_docs
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []
        
rag_retriever = RAGRetriever(vector_store, Embedding_Manager)

In [37]:
rag_retriever

In [38]:
rag_retriever.retrieve("what is the purpose of the document?")

Retrieving documents for query: 'what is the purpose of the document?' with top_k=5 and score_threshold=0.0
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embeddings generated successfully. Shape: (1, 384)
Retrieved 0 documents after applying score threshold.


[]

# Integration VectorDB Context pipeline With LLM output 